# Ch01-05: 데이터 변환과 정규화 이론 — 모범답안


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
np.random.seed(42)

---
## 문제 1 풀이

Box-Cox 프로파일 우도 + 95% CI

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from scipy import stats

np.random.seed(42); data = np.random.lognormal(2, 0.8, 500)

def pll(lam, y):
    n=len(y); yt=np.log(y) if abs(lam)<1e-10 else (y**lam-1)/lam
    s2=np.var(yt); return -n/2*np.log(s2)+(lam-1)*np.sum(np.log(y)) if s2>0 else -1e10

lams = np.linspace(-2,3,500); lls = np.array([pll(l,data) for l in lams])
bi = np.argmax(lls); lh = lams[bi]; llm = lls[bi]
thr = llm - stats.chi2.ppf(0.95,1)/2
ci = lams[lls>=thr]; ci_lo, ci_hi = ci[0], ci[-1]

fig,ax = plt.subplots(figsize=(10,5))
ax.plot(lams,lls,'b-',lw=2); ax.axhline(thr,color='red',ls='--')
ax.axvline(lh,color='green',lw=2,label=f'λ̂={lh:.3f}')
ax.axvspan(ci_lo,ci_hi,alpha=0.2,color='red',label=f'95% CI:[{ci_lo:.3f},{ci_hi:.3f}]')
ax.set_xlabel('λ'); ax.set_ylabel('프로파일 로그우도'); ax.legend()
plt.tight_layout(); plt.show()


**결과 해석**: 프로파일 우도 최대점이 최적 λ. CI에 0 포함 시 log 변환, 1 포함 시 변환 불필요 가능.

---
## 문제 2 풀이

분산 안정화 변환 검증

In [ ]:
import numpy as np, matplotlib.pyplot as plt
np.random.seed(42); nsim=10000
mus = np.arange(1,51)
vr = [np.var(np.random.poisson(m,nsim)) for m in mus]
vs = [np.var(np.sqrt(np.random.poisson(m,nsim))) for m in mus]
ps = np.linspace(0.05,0.95,30); nb=50
vp = [np.var(np.random.binomial(nb,p,nsim)/nb) for p in ps]
va = [np.var(np.arcsin(np.sqrt(np.random.binomial(nb,p,nsim)/nb))) for p in ps]

fig,axes = plt.subplots(2,2,figsize=(14,10))
axes[0,0].plot(mus,vr,'b-'); axes[0,0].plot(mus,mus,'r--'); axes[0,0].set_title('포아송: Var(Y)')
axes[0,1].plot(mus,vs,'g-'); axes[0,1].axhline(0.25,color='r',ls='--'); axes[0,1].set_title('포아송: Var(√Y)')
axes[1,0].plot(ps,vp,'b-'); axes[1,0].plot(ps,[p*(1-p)/nb for p in ps],'r--'); axes[1,0].set_title('이항: Var(p̂)')
axes[1,1].plot(ps,va,'g-'); axes[1,1].axhline(1/(4*nb),color='r',ls='--'); axes[1,1].set_title('이항: Var(arcsin√p̂)')
plt.tight_layout(); plt.show()


**결과 해석**: 포아송 √Y → 분산≈1/4, 이항 arcsin√p → 분산≈1/(4n). 델타 방법 예측과 시뮬레이션 일치.

---
## 문제 3 풀이

이상치 하 스케일러 비교

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, PowerTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.datasets import make_classification

np.random.seed(42)
X_c, y = make_classification(500, 10, n_informative=5, random_state=42)

def add_out(X, rate):
    Xn = X.copy(); no = int(len(X)*rate)
    if no>0: idx=np.random.choice(len(X),no,replace=False); Xn[idx]+=np.random.uniform(10,20,(no,X.shape[1]))*np.random.choice([-1,1],(no,X.shape[1]))
    return Xn

rates = [0,0.05,0.10,0.20]
scalers = {'Standard':StandardScaler(),'MinMax':MinMaxScaler(),'Robust':RobustScaler(),'Power':PowerTransformer('yeo-johnson')}
res = {n:[] for n in scalers}
for r in rates:
    Xd = add_out(X_c, r)
    for n, s in scalers.items():
        res[n].append(cross_val_score(LogisticRegression(max_iter=1000), s.fit_transform(Xd), y, cv=5).mean())

fig,ax = plt.subplots(figsize=(10,5))
for n,accs in res.items(): ax.plot([r*100 for r in rates], accs, 'o-', lw=2, label=n)
ax.set_xlabel('이상치 비율(%)'); ax.set_ylabel('정확도'); ax.legend(); ax.set_title('스케일러 비교')
plt.tight_layout(); plt.show()


**결과 해석**: RobustScaler가 이상치에 가장 강건. MinMaxScaler가 가장 민감.

---
## 문제 4 풀이

Fisher 정보량 vs 부트스트랩 CI

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import minimize_scalar

np.random.seed(42); n=200; data=np.random.lognormal(2,0.5,n)

def pll(lam,y):
    n=len(y); yt=np.log(y) if abs(lam)<1e-10 else (y**lam-1)/lam
    s2=np.var(yt); return -n/2*np.log(s2)+(lam-1)*np.sum(np.log(y)) if s2>0 else -1e10

lh = minimize_scalar(lambda l:-pll(l,data), bounds=(-2,3), method='bounded').x
eps=1e-4; fi = -(pll(lh+eps,data)-2*pll(lh,data)+pll(lh-eps,data))/eps**2
ase = 1/np.sqrt(fi) if fi>0 else np.nan

bl = [minimize_scalar(lambda l:-pll(l,data[np.random.choice(n,n)]), bounds=(-2,3), method='bounded').x for _ in range(2000)]
bl = np.array(bl); bse = bl.std()

ci_a = (lh-1.96*ase, lh+1.96*ase)
ci_p = (np.percentile(bl,2.5), np.percentile(bl,97.5))

fig,ax = plt.subplots(figsize=(10,5))
ax.hist(bl,50,density=True,alpha=0.6,label='부트스트랩')
xs = np.linspace(lh-4*ase,lh+4*ase,200)
ax.plot(xs,stats.norm.pdf(xs,lh,ase),'r-',lw=2,label='점근 정규')
ax.axvline(lh,color='green',lw=2); ax.legend()
ax.set_title(f'λ̂={lh:.3f}, 점근SE={ase:.4f}, 부트SE={bse:.4f}')
plt.tight_layout(); plt.show()

print(f"점근 CI: [{ci_a[0]:.4f},{ci_a[1]:.4f}]")
print(f"백분위 CI: [{ci_p[0]:.4f},{ci_p[1]:.4f}]")


**결과 해석**: Fisher SE와 부트스트랩 SE 유사 시 정규 근사 적절. BCa는 비대칭 분포에서도 정확한 커버리지 제공.

---
## 확장 토론

### 변환 선택 가이드

| 상황 | 변환 |
|------|------|
| 양수, 우편향 | Box-Cox |
| 음수 포함 | Yeo-Johnson |
| 카운트 | √y |
| 비율 | arcsin√p |
| 이상치 | Robust Scaling |